# <center> <img src="../img/ITESOLogo.png" alt="ITESO" width="480" height="130"> </center>
# <center> **Departamento de Electrónica, Sistemas e Informática** </center>
---
## <center> **Big Data** </center>
---
### <center> **Spring 2026** </center>
---
### <center> **Examples on Structured Streaming (sockets)** </center>
---
**Profesor**: Pablo Camarillo Ramirez

# Create SparkSession

In [8]:
import findspark
findspark.init()

In [9]:
from SparkUtils import SparkUtils

import pyspark.sql.functions as F

In [10]:
MASTER_URL = "spark://spark-master:7077"
APP_NAME = "Example: Structured Streaming"

spark = SparkUtils(MASTER_URL, APP_NAME)._spark

spark

# Create a data stream from a local socket

### Install netcat utility

In [11]:
!apt-get update
!apt-get install -y netcat

Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:2 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:3 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Reading package lists... Done
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
netcat is already the newest version (1.218-4ubuntu1).
0 upgraded, 0 newly installed, 0 to remove and 50 not upgraded.


### Connect Spark to the socket

In [14]:
# Create the remote connection
lines = spark.readStream \
    .format("socket") \
    .option("host", "localhost") \
    .option("port", 9999) \
    .load()

# Perform some transformations to the input data (word counter)
words = lines.select(F.explode(F.split(lines.value, " ")).alias("word"))

word_count = words.groupBy("word").count()

# Send transformed data to the Sink
query = word_count.writeStream \
    .outputMode("complete") \
    .format("console") \
    .start()

query.awaitTermination(300)

26/03/26 01:53:51 WARN TextSocketSourceProvider: The socket source should not be used for production applications! It does not support recovery.
26/03/26 01:53:51 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-390a0c88-bbde-4659-8510-f39f99351270. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/03/26 01:53:51 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


-------------------------------------------
Batch: 0
-------------------------------------------
+----+-----+
|word|count|
+----+-----+
+----+-----+



-------------------------------------------
Batch: 1
-------------------------------------------
+------+-----+
|  word|count|
+------+-----+
| world|    1|
|hello,|    1|
+------+-----+



26/03/26 01:55:34 WARN TextSocketMicroBatchStream: Stream closed by localhost:9999
ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/opt/spark/python/lib/py4j-0.10.9.9-src.zip/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/opt/spark/python/lib/py4j-0.10.9.9-src.zip/py4j/clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
  File "/usr/lib/python3.10/socket.py", line 705, in readinto
    return self._sock.recv_into(b)
KeyboardInterrupt


KeyboardInterrupt: 

In [ ]:
spark.stop()